In [1]:
# /// script
# requires-python = ">=3.10,<3.14"
# dependencies = [
#     "azure-identity>=1.19.0",
#     "azure-ai-projects>=2.0.0b2",
#     "openai>=1.109.0",
#     "python-dotenv>=1.0.0",
# ]
# ///
# Launch with one command (uv builds the environment + kernel for you):
#     uvx juv run m06_build_app_tour.ipynb
# Prereqs: `az login`, and a .env in this folder (see .env.example).
print("Ready. Everything below is keyless and safe to re-run.")

Ready. Everything below is keyless and safe to re-run.


# Lesson 6 · Build a generative application on Microsoft Foundry (hands-on tour)

Lesson 6 is a **build** lesson, so code is the honest surface here: this is where the real work happens, not in the portal. Three questions shape every generative app you ship on Foundry: **how do I connect, how do I call a model, and how do I reach everything else the app needs?**

This notebook answers all three against a live project, in the order you would actually write them.

- **Connect.** One `AIProjectClient`, built from the **project endpoint**.
- **Consume.** Call a deployed model, then a follow-up turn that remembers the first.
- **Integrate.** Enumerate the project's **connections**, the external services Foundry makes first-class.
- **Capstone.** Wrap the model in a small Foundry **agent**, the thing you actually ship. Lessons 10 and 11 go deep.

Two properties make this safe to lean on:

- **Keyless.** Everything authenticates with your `az login` identity through Microsoft Entra ID. No API keys anywhere.
- **Idempotent.** Every cell either reads, or converges to the same state, so you can run the whole thing top to bottom as many times as you like.

## Setup: one credential, one config source

`AzureCliCredential` uses exactly the identity from `az login`. We prefer it over `DefaultAzureCredential` in demos because `DefaultAzureCredential` probes environment variables first and can quietly select a different identity than the one you signed in with. Deterministic beats convenient when a camera is rolling.

Config comes from `.env` (copy `.env.example`), never hardcoded. `MODEL_DEPLOYMENT_NAME` holds the **deployment** name you chose, not the model family name.

In [2]:
# --- Setup: one credential, read config from .env -------------------------------------
import os
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)   # this lesson's .env wins over any ambient vars
cred = AzureCliCredential()  # your `az login` identity

PROJECT_EP = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
DEPLOYMENT = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5-mini")
print("project    :", PROJECT_EP)
print("deployment :", DEPLOYMENT)

project    : https://aif-ai103-mvp-quc6kxy2j2v2q.services.ai.azure.com/api/projects/proj-lesson-06
deployment : gpt-5-mini


## 1 · Connect, the front door

The endpoint is the punchline of this whole lesson. It is the **project** URL:

`https://<account>.services.ai.azure.com/api/projects/<project>`

Not a model endpoint. That single address reaches every **deployment** and every **connection** in the project, which is why one `AIProjectClient` is all the plumbing your application needs.

Notice the shape of the health check below: we list the project's model deployments at startup. If the URL or the identity is wrong, the failure surfaces **here**, not halfway through a user's request. Fail fast, on your own terms.

In [3]:
# CONNECT: one client, built from the PROJECT endpoint, keyless.
from azure.ai.projects import AIProjectClient

project = AIProjectClient(endpoint=PROJECT_EP, credential=cred)

# Startup health check: enumerate the models deployed in this project.
names = [d.name for d in project.deployments.list()]
print("connected:", PROJECT_EP)
print("deployments reachable from this one client:", names or "(none yet)")

connected: https://aif-ai103-mvp-quc6kxy2j2v2q.services.ai.azure.com/api/projects/proj-lesson-06
deployments reachable from this one client: ['gpt-5-mini', 'text-embedding-3-small']


## 2 · Consume the deployed model

The **deployment** itself is infrastructure, defined as code in `infra/ai103-core.mvp.bicep` with its SKU, its pinned model version, and its `raiPolicyName`. Here we do the other half of the job: **consume** it.

`project.get_openai_client()` returns a standard **OpenAI client**, already bearer-token authenticated against `https://ai.azure.com/.default`. The code below is the same OpenAI code you already know. Only now it is keyless and pointed at your Foundry project. In CI you swap the one credential line for a federated (OIDC) identity and it still ships secretless.

In [4]:
# CONSUME (turn 1): a standard OpenAI client, keyless, obtained from the project.
oai = project.get_openai_client()

# Ask about a concept the model knows cold. Never ask a model to define the platform
# hosting it: "what is Microsoft Foundry?" reliably hallucinates, because the product
# is newer than the training data. That is a groundedness lesson (Lessons 3 and 5),
# and it has no business derailing a connectivity beat.
first = oai.responses.create(
    model=DEPLOYMENT,
    input="In one sentence, what is retrieval-augmented generation?",
)
print("Q1: What is retrieval-augmented generation?")
print("A1:", first.output_text.strip())

Q1: What is retrieval-augmented generation?
A1: Retrieval-augmented generation (RAG) is a method that improves a generative language model by first retrieving relevant external documents or passages and then conditioning the model's output on those retrieved pieces to produce more accurate, grounded, and up-to-date responses.


### 2b · A follow-up turn that remembers

`previous_response_id` threads this turn onto the previous one. That is conversation **memory** with zero extra plumbing: no message array to marshal by hand, no state store to operate. Watch the pronoun. We ask what "it" is *not*, and never say what "it" is.

In [5]:
# CONSUME (turn 2): previous_response_id carries the first turn's context forward.
second = oai.responses.create(
    model=DEPLOYMENT,
    input="Now name one thing it is NOT, in one sentence.",
    previous_response_id=first.id,
)
print("Q2 (follow-up): name one thing it is NOT")
print("A2:", second.output_text.strip())

Q2 (follow-up): name one thing it is NOT
A2: It is not the same as fine-tuning a model — RAG dynamically fetches external documents at inference time rather than embedding all knowledge into the model's weights.


## 3 · Integrate through connections

A real generative app needs more than a model: a search index for retrieval, Application Insights for telemetry, a storage account for documents. Foundry makes those external services **first-class objects** on the project, called **connections**.

Define a connection once, in the portal or in Bicep, and every app in the project reuses it by name. The alternative is the thing this replaces: endpoints and secrets copy-pasted across a dozen config files.

An empty list below is a valid result, not a bug. It means nothing has been wired into this project yet.

In [6]:
# INTEGRATE: enumerate the project's connections (its connectors).
found = False
for conn in project.connections.list():
    found = True
    ctype = getattr(conn, "type", None) or getattr(conn, "connection_type", "?")
    print(f"  - {getattr(conn, 'name', '?'):<28} type={ctype}")
if not found:
    print("  (no connections yet: add one in the portal or via Bicep, see infra/modules/*.bicep)")
print("\nDefine a connection once, reuse it everywhere. No secrets scattered in code.")

  - l06-appinsights              type=ConnectionType.APPLICATION_INSIGHTS

Define a connection once, reuse it everywhere. No secrets scattered in code.


## 4 · Capstone, wrap the model in an agent

An **agent** is a model plus instructions, and later, tools and memory, that you publish once and call by name. That is the generative *application*, as opposed to the raw model call you just made. Lessons 10 and 11 go deep. This is a sixty-second taste.

Three notes for the road:

- This is a **preview** surface. The cell catches its own exceptions and prints a skip, so a churning API never derails a live demo.
- The response references the agent through the `agent_reference` key. The older `agent` key is deprecated, and that swap is a live churn point today.
- The `finally` block deletes the version we created, even on failure. That is what keeps the notebook **idempotent**: re-run it and nothing accumulates.

In [7]:
# CAPSTONE: create an agent version, run one turn through it, then always clean up.
agent = None
try:
    from azure.ai.projects.models import PromptAgentDefinition

    agent = project.agents.create_version(
        agent_name="m06-hello-agent",
        definition=PromptAgentDefinition(
            model=DEPLOYMENT,
            instructions="You are a concise Azure AI tutor. Answer in one short sentence.",
        ),
    )
    print(f"created agent '{agent.name}' version {agent.version}")

    conv = oai.conversations.create(
        items=[{"type": "message", "role": "user",
                "content": "Give me a one-sentence definition of a vector embedding."}],
    )
    # The response references the agent by name. The key is `agent_reference`;
    # the older `agent` key is deprecated. A real churn point in this preview API.
    resp = oai.responses.create(
        conversation=conv.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="",
    )
    print("agent says:", resp.output_text.strip())
    oai.conversations.delete(conversation_id=conv.id)
except Exception as exc:  # preview surface: narrate the skip, never crash the notebook
    print(f"SKIP: agent capstone unavailable here ({type(exc).__name__}: {exc}).")
    print("The connect, consume, and connections beats above are the stable core of this demo.")
finally:
    # Delete the version we created even if the call above failed, so re-running
    # this notebook never leaves orphaned agents behind.
    if agent is not None:
        try:
            project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
            print("cleaned up: agent version deleted (idempotent).")
        except Exception:
            pass

created agent 'm06-hello-agent' version 1


agent says: A vector embedding is a numeric vector that encodes an object's semantic and relational properties in a continuous space.


cleaned up: agent version deleted (idempotent).


## Recap

One project client, three answers, and the shape of a real application:

- **Connect.** The project endpoint is the single front door, and listing deployments at startup turns a bad URL or a bad identity into an immediate, obvious failure.
- **Consume.** `get_openai_client()` gives you ordinary OpenAI code, keyless, and `previous_response_id` gives you conversation memory for free.
- **Integrate.** Connections turn external services into named, reusable objects instead of secrets scattered across config files.
- **Ship.** An agent is a model plus instructions, published and callable by name. That is Lessons 10 and 11.

Everything here was keyless and idempotent. That combination is exactly what lets the same code run unattended in a scheduled job or a CI pipeline, with a federated (OIDC) identity swapped in for `AzureCliCredential` and not one other line changed.